# Tier 4.0 — LLM-as-a-Judge Re-Ranking (Azure GPU)

**BSARD RAG Thesis | RQ1 | T4.0 Experiments**

## Before running — one-time setup

1. **GPU compute instance**: Azure ML → Compute → Create → `Standard_NC6ads_A10_v4` (preferred) or `NC4as_T4_v3`
2. **Set Cell 0** with your `GITHUB_TOKEN` and `AZURE_CONTAINER_SAS_URL`
3. Run cells top to bottom

### How to generate the Container SAS URL
Azure Portal → Storage Accounts → *your account* → Containers → `bsard-data`
→ `...` → **Generate SAS** → Permissions: **Read + List + Write** → Expiry: 1 year
→ Generate → copy the **Blob SAS URL** (full `https://...` URL, not just the token)

## Current state

**All results need to be re-run** with the correct first-stage retriever (`bm25_tuned_k11.5_b0.25`).
Previous runs used `hybrid_rrf_k60` as first stage — those results are superseded.

Results write to `output/results/agentic/llm_judge/llm_rerank/` (binary) and
`output/results/agentic/llm_judge/0to10/` (0–10 numeric scoring). `output/` is gitignored — on Azure,
Cell 13 uploads results to Azure Blob Storage; download them locally via the blob.

## Expected execution times (T4 GPU)

| Phase | Expected time |
|---|---|
| Setup (Cells 0–7) | ~20 min |
| TEST experiment (Cell 10) | ~2–3h |
| Significance test (Cell 11) | ~5 min (BM25 baseline, no LLM) |
| 0–10 numeric test run (Cell 11b) | ~4–5h |
| Upload to blob (Cell 13) | ~1 min |

In [ ]:
# ── Cell 0: Configuration ─────────────────────────────────────────────────────
# Set GITHUB_TOKEN and AZURE_CONTAINER_SAS_URL before running any other cell.

GITHUB_TOKEN = ''
# How to get:
#   github.com → Settings → Developer settings
#   → Personal access tokens → Tokens (classic) → New token → scope: repo → Generate

AZURE_CONTAINER_SAS_URL = ''
# How to get:
#   Azure Portal → Storage Accounts → your account
#   → Containers → bsard-data → (...) → Generate SAS
#   → Permissions: Read + List → Expiry: 1 year → Generate
#   → Copy the full "Blob SAS URL" (starts with https://...)

REPO     = 'MariusPasch/bsard-rag-thesis'
CLONE_DIR = '/home/azureuser/repo'                # mono-repo clone root
REPO_DIR  = f'{CLONE_DIR}/RQ1_Retrieval_Methods'  # RQ1 component root

assert GITHUB_TOKEN,            'Set GITHUB_TOKEN above before running!'
assert AZURE_CONTAINER_SAS_URL, 'Set AZURE_CONTAINER_SAS_URL above before running!'
print('Config OK')


In [ ]:
# # ── Cell 0b: Reset — delete result files so all experiment cells re-run ───────
# # Run this cell ONLY if you want a full fresh run.
# import os
# from pathlib import Path

# files_to_delete = [
#     f'{REPO_DIR}/output/results/agentic/llm_judge/llm_rerank/llm_rerank_binary_top10_val.json',
#     f'{REPO_DIR}/output/results/agentic/llm_judge/llm_rerank/llm_rerank_binary_top20_val.json',
#     f'{REPO_DIR}/output/results/agentic/llm_judge/llm_rerank/llm_rerank_binary_top50_val.json',
#     f'{REPO_DIR}/output/results/agentic/llm_judge/llm_rerank/llm_rerank_binary_top50_test.json',
#     f'{REPO_DIR}/output/results/agentic/llm_judge/0to10/llm_rerank_0to10_top50_test.json',
# ]

# for p in files_to_delete:
#     path = Path(p)
#     if path.exists():
#         path.unlink()
#         print(f'  Deleted: {path.name}')
#     else:
#         print(f'  Not found (skip): {path.name}')

# print('\nReset complete — all experiment cells will now run from scratch.')

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.stdout.strip():
    print('GPU:', result.stdout.strip())
    print('GPU OK')
else:
    print('WARNING: No GPU detected!')
    print('  Azure ML: ensure compute instance uses NC4as_T4_v3 or NC6ads_A10_v4')
    print(result.stderr)


In [ ]:
# ── Cell 2: Install Ollama and pull llama3.1:8b (~10 min on first run) ────────
import json, os, subprocess, time, urllib.request

OLLAMA_LOG = '/tmp/ollama_server.log'

def model_available() -> bool:
    try:
        with urllib.request.urlopen('http://localhost:11434/api/tags', timeout=5) as r:
            return any('llama3.1' in m['name'] for m in json.loads(r.read()).get('models', []))
    except Exception:
        return False

if not os.path.exists('/usr/local/bin/ollama'):
    print('Installing Ollama via official script...')
    subprocess.run(
        ['bash', '-c', 'curl -fsSL https://ollama.com/install.sh | sh'],
        check=True
    )
    print('Ollama installed.')
else:
    print('Ollama already installed.')

if not model_available():
    print('Starting Ollama server...')
    subprocess.Popen(
        ['ollama', 'serve'],
        env={
            **os.environ,
            'HOME': '/root',
            'OLLAMA_NUM_GPU': '99',
            'OLLAMA_FLASH_ATTENTION': '1',
            'OLLAMA_HOST': '0.0.0.0:11434',
        },
        stdout=open(OLLAMA_LOG, 'w'),
        stderr=subprocess.STDOUT,
    )
    time.sleep(8)
    if not model_available():
        print('Pulling llama3.1:8b (~4.7 GB, ~5-10 min)...')
        subprocess.run(['ollama', 'pull', 'llama3.1:8b'], check=True)
        time.sleep(3)

resp   = urllib.request.urlopen('http://localhost:11434/api/tags', timeout=10)
models = [m['name'] for m in json.loads(resp.read()).get('models', [])]
print('Available models:', models)
assert any('llama3.1' in m for m in models), 'llama3.1:8b not found!'
print('Ollama ready.')


In [ ]:
# ── Cell 3: Download data from Azure Blob Storage ─────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'azure-storage-blob'], check=True)

from azure.storage.blob import ContainerClient
from pathlib import Path

OUTPUT_DIR = Path(REPO_DIR) / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# BM25-only: no embedding files needed
downloads = {
    'bsard_articles_dedup.parquet': OUTPUT_DIR,
    'bsard_corpus.db':              OUTPUT_DIR,
}

client = ContainerClient.from_container_url(AZURE_CONTAINER_SAS_URL)

for blob_name, dest_dir in downloads.items():
    dest_path = Path(dest_dir) / blob_name
    if dest_path.exists():
        print(f'  Already exists: {blob_name} ({dest_path.stat().st_size / 1e6:.1f} MB)')
        continue
    print(f'  Downloading {blob_name} ...', end='', flush=True)
    with open(dest_path, 'wb') as f:
        client.get_blob_client(blob_name).download_blob().readinto(f)
    print(f' done ({dest_path.stat().st_size / 1e6:.1f} MB)')

print('\nAll data files ready.')


In [ ]:
# import subprocess, shutil, os

# # Remove the broken directory
# shutil.rmtree('/home/azureuser/repo', ignore_errors=True)
# print('Removed /home/azureuser/repo')

In [ ]:
# import subprocess, os

# os.makedirs('/home/azureuser/repo', exist_ok=True)
# os.chdir('/home/azureuser')  # reset cwd to a valid directory

# r = subprocess.run(
#     ['git', 'clone', f'https://{GITHUB_TOKEN}@github.com/{REPO}.git', REPO_DIR],
#     capture_output=True, text=True, timeout=120, cwd='/home/azureuser'
# )
# print(r.stdout)
# print(r.stderr)


In [ ]:
# ── Cell 4: Clone GitHub repo ─────────────────────────────────────────────────
import os, subprocess

if os.path.exists(CLONE_DIR):
    print('Repo already cloned — pulling latest...')
    subprocess.run(
        ['git', '-C', CLONE_DIR, 'remote', 'set-url', 'origin',
         f'https://{GITHUB_TOKEN}@github.com/{REPO}.git'],
        capture_output=True, text=True
    )
    r = subprocess.run(['git', '-C', CLONE_DIR, 'pull'], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())
else:
    print(f'Cloning {REPO}...')
    r = subprocess.run(
        ['git', 'clone', f'https://{GITHUB_TOKEN}@github.com/{REPO}.git', CLONE_DIR],
        capture_output=True, text=True, timeout=120
    )
    if r.returncode != 0:
        print('STDERR:', r.stderr[-1000:])
        raise RuntimeError('git clone failed')

os.chdir(REPO_DIR)
branch = subprocess.run(['git', 'branch', '--show-current'],
                        capture_output=True, text=True).stdout.strip()
print(f'Working directory: {os.getcwd()}  |  branch: {branch}')

In [ ]:
# import subprocess, sys

# r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'spacy'],
#                    capture_output=True, text=True)
# print('spacy install:', r.returncode, r.stderr[-200:] if r.returncode != 0 else 'OK')

# r2 = subprocess.run([sys.executable, '-m', 'spacy', 'download', 'fr_core_news_lg'],
#                     capture_output=True, text=True)
# print('fr_core_news_lg:', r2.returncode)
# print(r2.stdout[-300:])
# print(r2.stderr[-300:])

In [ ]:
# ── Cell 5: Install Python dependencies (~5 min) ──────────────────────────────
import subprocess, sys, os

cuda_out = subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout
cuda_ver = 'cu118' if 'release 11' in cuda_out else 'cu121'
print(f'CUDA detected → torch variant: {cuda_ver}')

cmds = [
    ([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements.txt'],
     'requirements.txt'),
    ([sys.executable, '-m', 'pip', 'install', '-q', 'azure-storage-blob'],
     'azure-storage-blob'),
    ([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
      '--index-url', f'https://download.pytorch.org/whl/{cuda_ver}'],
     'torch'),
    ([sys.executable, '-m', 'pip', 'install', '-q',
      'langgraph>=0.1.0', 'langchain-core>=0.2.0', 'requests'],
     'langgraph + langchain-core + requests'),
    # tf-keras: transformers tries to import Keras 3 (which breaks) — this shim fixes it
    ([sys.executable, '-m', 'pip', 'install', '-q', 'tf-keras'],
     'tf-keras'),
    # timm: ImageNetInfo requires >=0.9.2; base env ships an older version
    ([sys.executable, '-m', 'pip', 'install', '-q', 'timm>=0.9.2'],
     'timm>=0.9.2'),
]
for cmd, label in cmds:
    print(f'  {label} ...', end='', flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(' OK' if r.returncode == 0 else f' WARN({r.returncode})')
    if r.returncode != 0:
        print(r.stderr[-200:])

# bsard_evaluation — editable local package from the RQ3 repo (required by evaluation/runner.py)
# bsard_evaluation lives in the same mono-repo (RQ3_Autonomous_Evaluation),
# already cloned above — just install it editable.
RQ3_DIR = f'{CLONE_DIR}/RQ3_Autonomous_Evaluation'
print('  bsard_evaluation ...', end='', flush=True)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', RQ3_DIR],
                   capture_output=True, text=True)
print(' OK' if r.returncode == 0 else f' WARN({r.returncode})\n{r.stderr[-200:]}')

# spaCy French model — build direct pip URL to avoid `spacy download` 404 issues
import spacy as _spacy
_sv = _spacy.__version__
_base = 'https://github.com/explosion/spacy-models/releases/download'
_whl  = f'fr_core_news_lg-{_sv}/fr_core_news_lg-{_sv}-py3-none-any.whl'
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', f'{_base}/{_whl}'],
    capture_output=True, text=True
)
if r.returncode != 0:
    # Fallback: try major.minor.0 in case patch differs
    _sv2 = '.'.join(_sv.split('.')[:2]) + '.0'
    _whl2 = f'fr_core_news_lg-{_sv2}/fr_core_news_lg-{_sv2}-py3-none-any.whl'
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', f'{_base}/{_whl2}'],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        raise RuntimeError(f'spaCy fr_core_news_lg install failed:\n{r.stderr[-400:]}')

import spacy
spacy.load('fr_core_news_lg')
print(f'spaCy {spacy.__version__} OK — fr_core_news_lg loaded')

In [ ]:
# ── Cell 6: Pre-flight checks ─────────────────────────────────────────────────
import os, sys, requests as _req
from pathlib import Path

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# Ollama
r = _req.get('http://localhost:11434/api/tags', timeout=5)
models = [m['name'] for m in r.json().get('models', [])]
print('Ollama models:', models)
assert any('llama3.1' in m for m in models), 'llama3.1:8b not found!'

# Fewshot examples
assert Path('evaluation/data/fewshot_examples.json').exists(), 'fewshot_examples.json missing!'
print('fewshot_examples.json: OK')

# Data files — BM25 only (no embeddings needed)
for f in [
    'output/bsard_articles_dedup.parquet',
    'output/bsard_corpus.db',
]:
    assert Path(f).exists(), f'{f} missing!'
    print(f'  {f}: OK')

print('\nAll checks passed.')


In [ ]:
# ── Cell 7: LLM latency benchmark — confirm GPU speed ────────────────────────
# Used to estimate experiment times. Also verifies flash attention is active.
import time, os, sys, subprocess
import requests as _req

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

from retrieval.agentic.llm_eval_prompts import (
    load_fewshot_examples, format_fewshot_block, LLM_JUDGE_BINARY_PROMPT
)

def benchmark_call(article_words: int, label: str) -> float:
    examples = load_fewshot_examples()
    fewshot  = format_fewshot_block(examples, 'binary')
    prompt   = LLM_JUDGE_BINARY_PROMPT.format(
        fewshot_block=fewshot,
        question='Quelle est la peine pour vol simple ?',
        article_text_truncated=' '.join(['mot'] * article_words),
    )
    t0  = time.perf_counter()
    r   = _req.post('http://localhost:11434/api/generate',
                    json={'model': 'llama3.1:8b', 'prompt': prompt, 'stream': False,
                          'options': {'temperature': 0.0, 'num_predict': 4}},
                    timeout=300).json()
    elapsed = time.perf_counter() - t0
    n_in = r.get('prompt_eval_count', 0)
    print(f'  [{label}] {article_words} words → {n_in} tokens | '
          f'{elapsed:.2f}s | {n_in/elapsed:.0f} tok/s | response={r["response"]!r}')
    return elapsed

print('Benchmarking (warmup + 3 lengths)...')
benchmark_call(200, 'warmup')
e_300  = benchmark_call(300,  '300-tok')
e_1000 = benchmark_call(1000, '1000-tok')

print('\n--- Time estimates for significance test (222 test questions) ---')
# Significance test re-scores top-20 articles per question
for label, e in [('300-tok  top-20', e_300), ('1000-tok top-20', e_1000)]:
    h = 222 * 20 * e / 3600
    print(f'  {label}: ~{h:.1f}h for T4.0 re-score')
print('T3-C CE reranker: ~5-10 min (independent of token count)')

print('\nNote: Val experiments are not run (hyperparameters fixed a priori — plan §3.2).')
print('Test/0–10 numeric cells skip automatically if result files already exist in output/results/.')

## Cell 8: Val experiments — not run

Val experiments (`binary_top10/20/50/100_val`, `0to10_top50_val`, `cot_top50_val`) are **not run**.
Hyperparameters (`BEST_TOP_N=50`, `BEST_VARIANT='binary'`) are fixed a priori per plan §3.2 — no empirical val tuning.
Proceed directly to Cell 9 to confirm the canonical config.

In [ ]:
# ── Cell 9: Canonical hyperparameters (fixed a priori — plan §3.2) ───────────
# Top-N and scoring variant are pre-specified design decisions, not empirically tuned on val.
# These values are used directly in Cell 10 (test run) and Cell 11 (significance test).
BEST_TOP_N   = 50       # pre-specified: good coverage without scoring low-probability candidates
BEST_VARIANT = 'binary' # pre-specified: 8B-class models produce poorly calibrated continuous scores
print(f'Canonical config: variant={BEST_VARIANT}  top_n={BEST_TOP_N}')

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm>=0.9.2'], check=True); print('OK')


In [ ]:
# ── Cell 10: TEST experiment ──────────────────────────────────────────────────
# BEST_TOP_N and BEST_VARIANT set in Cell 9 (default: binary, top_n=50).
# Skips automatically if result file already exists.
import json, subprocess, sys, os
from pathlib import Path

os.chdir(REPO_DIR)

RESULTS_DIR = Path(f'{REPO_DIR}/output/results/agentic/llm_judge/llm_rerank')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
TEST_JSON   = RESULTS_DIR / f'llm_rerank_{BEST_VARIANT}_top{BEST_TOP_N}_test.json'

if TEST_JSON.exists():
    d  = json.loads(TEST_JSON.read_text())
    m  = d['metrics']
    hp = d.get('hyperparameters', {})
    print(f'{TEST_JSON.name} already exists — skipping.')
    print(f"  experiment_id = {d['experiment_id']}")
    print(f"  top_n={hp.get('top_n')}  variant={hp.get('prompt_variant')}  "
          f"first_stage={hp.get('first_stage')}")
    print(f"  R@10={m['Recall@10']:.4f}  R@100={m['Recall@100']:.4f}  MRR@10={m['MRR@10']:.4f}")
else:
    print(f'Running test: variant={BEST_VARIANT}, top_n={BEST_TOP_N} (~2–3h)...')
    result = subprocess.run(
        [sys.executable, '-u', 'scripts/evaluation/tier3/run_llm_rerank_experiments.py',
         '--split', 'test',
         '--best-variant', BEST_VARIANT,
         '--best-top-n', str(BEST_TOP_N),
         '--max-article-tokens', '1000'],
        cwd=REPO_DIR,
    )
    print('Exit code:', result.returncode)

In [ ]:
# import subprocess, sys
# subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tf-keras'], check=True)
# print('tf-keras installed — now restart the kernel and re-run from Cell 11')


In [ ]:
# ── Cell 11: Post-hoc significance test (T4.0 vs T1 BM25 baseline) ────────────
# Compares T4.0 LLM reranker vs BM25 alone (no reranking) on same backbone.
# BM25 baseline runs in <1 min (no LLM calls). T4.0 per-query recalls are
# loaded from the saved result JSON (put there by the run script). If absent,
# T4.0 is re-scored from cache (~10–30 min depending on cache hit rate).
# Result: p-value patched into llm_rerank_binary_top50_test.json.
import json, os, sys, time
import numpy as np
from pathlib import Path
from scipy.stats import ttest_rel

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

from evaluation.split import load_questions, make_ground_truth
from retrieval.sparse import BM25Retriever
import pandas as pd

CORPUS_PATH = Path('output/bsard_articles_dedup.parquet')
DB_PATH     = Path('output/bsard_corpus.db')
RESULTS_DIR = Path(f'{REPO_DIR}/output/results/agentic/llm_judge/llm_rerank')
T40_PATH    = RESULTS_DIR / f'llm_rerank_{BEST_VARIANT}_top{BEST_TOP_N}_test.json'

if not T40_PATH.exists():
    print(f'T4.0 result not found at {T40_PATH} — run Cell 10 first.')
else:
    def per_query_recall(results, ground_truth, k):
        scores = []
        for qid, relevant in ground_truth.items():
            if not relevant:
                continue
            retrieved = results.get(qid, [])[:k]
            scores.append(len(set(retrieved) & set(relevant)) / len(relevant))
        return scores

    print('Loading corpus and test questions...')
    corpus    = pd.read_parquet(CORPUS_PATH)
    questions = load_questions(DB_PATH, subset='test')
    gt        = make_ground_truth(questions)
    print(f'  {len(questions)} test questions | {len(corpus)} articles')

    t40_result = json.loads(T40_PATH.read_text(encoding='utf-8'))

    # ── T4.0 per-query recalls ────────────────────────────────────────────────
    if 'per_query_recalls' in t40_result:
        r40_k10  = t40_result['per_query_recalls']['k10']
        r40_k100 = t40_result['per_query_recalls']['k100']
        print('T4.0 per_query_recalls loaded from result JSON.')
    else:
        print('per_query_recalls not in result — re-scoring T4.0 from cache...')
        from retrieval.llm_reranker import LLMJudgeReranker
        from retrieval.agentic.llm_client import OllamaClient
        article_texts = dict(zip(corpus['article_id'], corpus['article_text']))
        qid_map       = {q['question_text']: q['question_id'] for q in questions}
        bm25 = BM25Retriever(corpus, variant='okapi', normalization='lemmatize',
                             field_weighting='text_only', k1=1.5, b=0.25)
        cache_path = Path(f'{REPO_DIR}/output/llm_judge_cache_binary_test_tok1000.json')
        t40 = LLMJudgeReranker(
            first_stage_retriever=bm25, article_texts=article_texts,
            llm_client=OllamaClient(), top_n=BEST_TOP_N, max_article_tokens=1000,
            prompt_variant='binary', cache_path=cache_path,
            question_id_fn=lambda q: qid_map.get(q, hash(q) & 0x7FFFFFFF),
            fewshot_path=Path('evaluation/agentic/fewshot_examples.json'),
        )
        t40_res = {}
        for i, q in enumerate(questions, 1):
            ids, _ = t40.retrieve(q['question_text'], top_k=100)
            t40_res[q['question_id']] = ids
            if i % 30 == 0 or i == len(questions):
                print(f'  [{i}/{len(questions)}]')
        t40.save_cache()
        r40_k10  = per_query_recall(t40_res, gt, k=10)
        r40_k100 = per_query_recall(t40_res, gt, k=100)

    # ── BM25 baseline (T1, no reranking) ─────────────────────────────────────
    print('\nBuilding BM25 baseline (bm25_tuned_k11.5_b0.25)...')
    bm25_baseline = BM25Retriever(
        corpus, variant='okapi', normalization='lemmatize',
        field_weighting='text_only', k1=1.5, b=0.25,
    )
    bm25_res = {}
    t0 = time.perf_counter()
    for q in questions:
        ids, _ = bm25_baseline.retrieve(q['question_text'], top_k=100)
        bm25_res[q['question_id']] = ids
    print(f'  BM25 baseline done in {(time.perf_counter()-t0)*1000:.0f}ms')

    rbm25_k10  = per_query_recall(bm25_res, gt, k=10)
    rbm25_k100 = per_query_recall(bm25_res, gt, k=100)

    # ── Paired t-test ─────────────────────────────────────────────────────────
    _, p10  = ttest_rel(r40_k10,  rbm25_k10)
    _, p100 = ttest_rel(r40_k100, rbm25_k100)

    print(f'\n{"="*55}')
    print(f'T4.0 (LLM rerank, top_n={BEST_TOP_N})  mean R@10 = {np.mean(r40_k10):.4f}')
    print(f'BM25 baseline (T1, no reranking)         R@10 = {np.mean(rbm25_k10):.4f}')
    print(f'Delta                                         = {np.mean(r40_k10) - np.mean(rbm25_k10):+.4f}')
    print(f'p-value R@10  = {p10:.4f}  → {"SIGNIFICANT (p<0.05)" if p10 < 0.05 else "not significant"}')
    print(f'p-value R@100 = {p100:.4f}')

    # ── Patch result JSON ─────────────────────────────────────────────────────
    t40_result['significance_vs_anchor'] = {
        'anchor_experiment_id': 'bm25_tuned_k11.5_b0.25',
        'p_value_recall10':  round(float(p10),  4),
        'p_value_recall100': round(float(p100), 4),
        'significant':       bool(p10 < 0.05),
    }
    T40_PATH.write_text(json.dumps(t40_result, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'\nPatched {T40_PATH.name} with significance_vs_anchor.')

In [ ]:
# ── Cell 11b: 0–10 numeric scoring test run (appendix — interpretability) ─────────────────────
# Runs llm_rerank_0to10_top50_test at top_n=50 (consistent with canonical binary result).
# First stage: bm25_tuned_k11.5_b0.25 (BM25 Okapi, k1=1.5, b=0.25, lemmatize, text_only)
# Expected runtime: ~3h on T4 GPU. Skips automatically if file already exists.
#
# Prompt fix: Score label comes before Explication so the score number Score label comes before Explication so the score number
# is always generated within the 64-token budget.
#
# System prompt fix: replaces the generic "scoring system" framing with a legal-domain
# context that establishes regulatory analysis before the article text appears. This
# prevents LLaMA 3.1 8B's content safety filter from refusing to score articles from
# criminal, drug, weapons, and sexual-offense law (legitimate Belgian statutory content).
# Observed parse failure rate dropped from 18.2% (an earlier prompt version where token budget ran out before the score digit was emitted) to 3.2%
# (generic system prompt); this fix targets the residual 3.2%.
# Model remains llama3.1:8b — no model change.
#
# Interruption-safe: score cache is checkpointed every 40 questions (~40 min apart).

import json, os, sys, time
from pathlib import Path

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

RESULTS_DIR = Path(f'{REPO_DIR}/output/results/agentic/llm_judge/0to10')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
NUM10_JSON    = RESULTS_DIR / 'llm_rerank_0to10_top50_test.json'

if NUM10_JSON.exists():
    d = json.loads(NUM10_JSON.read_text())
    m = d['metrics']
    print(f'{NUM10_JSON.name} already exists — skipping.')
    print(f"  R@10={m.get('Recall@10', 0):.4f}  R@20={m.get('Recall@20', 0):.4f}  "
          f"MRR@10={m.get('MRR@10', 0):.4f}")
else:
    import numpy as np
    import pandas as pd
    from evaluation.split import load_questions
    from evaluation.stratify import load_strata
    from evaluation.runner import run_experiment, save_result
    from retrieval.sparse import BM25Retriever
    from retrieval.llm_reranker import LLMJudgeReranker
    from retrieval.agentic.llm_client import OllamaClient

    import retrieval.agentic.llm_eval_prompts as _prompts
    _prompts.LLM_JUDGE_0TO10_PROMPT = (
        "Question : {question}\n\n"
        "Passage : {article_text_truncated}\n\n"
        "Donnez un score de 0 à 10, puis expliquez brièvement pourquoi ce passage est ou n'est pas pertinent.\n\n"
        "Score (0-10) :"
    )

    CORPUS_PATH = Path('output/bsard_articles_dedup.parquet')
    DB_PATH     = Path('output/bsard_corpus.db')
    NUM10_CACHE   = Path('output/llm_judge_cache_0to10_test_tok1000.json')
    TOP_N       = 50
    CACHE_CHECKPOINT_EVERY = 40

    print('Loading corpus and questions...')
    corpus        = pd.read_parquet(CORPUS_PATH)
    article_texts = dict(zip(corpus['article_id'], corpus['article_text']))
    questions     = load_questions(DB_PATH, subset='test')
    qid_map       = {q['question_text']: q['question_id'] for q in questions}
    print(f'  {len(questions)} test questions | {len(article_texts)} articles')

    strata = load_strata()
    if strata is None:
        raise RuntimeError('query_strata.json not found — run evaluation/stratify.py first')

    t_idx0 = time.perf_counter()
    print('\nBuilding first-stage retriever (bm25_tuned_k11.5_b0.25)...')
    first_stage = BM25Retriever(
        corpus, variant='okapi', normalization='lemmatize',
        field_weighting='text_only', k1=1.5, b=0.25,
    )
    index_build_s = time.perf_counter() - t_idx0
    print(f'  Index built in {index_build_s:.1f}s')

    llm_client = OllamaClient()

    # ── System prompt fix ────────────────────────────────────────────────────
    # The legal-domain framing establishes regulatory context before the article
    # text is presented, preventing the safety filter from refusing to score
    # legitimate Belgian statutory content (criminal penalties, drug law, etc.).
    _LEGAL_SYSTEM_PROMPT = (
        "Tu es un assistant de recherche juridique spécialisé dans l'analyse du droit belge. "
        "Les passages ci-dessous proviennent exclusivement de textes législatifs officiels belges "
        "(codes, lois, arrêtés royaux). "
        "Ta tâche est d'évaluer la pertinence d'un article de loi pour une question juridique "
        "dans un contexte académique. "
        "Réponds uniquement avec un entier de 0 à 10. Aucun autre texte."
    )

    _orig_generate = llm_client.generate
    def _generate_with_system(prompt, temperature=0.0, max_tokens=128, timeout=1800):
        import time as _time
        payload = {
            "model":  llm_client.model,
            "system": _LEGAL_SYSTEM_PROMPT,
            "prompt": prompt, "stream": False,
            "options": {"temperature": temperature, "num_predict": max_tokens},
        }
        t0 = _time.perf_counter()
        resp = llm_client._session.post(
            f"{llm_client.base_url}/api/generate", json=payload, timeout=timeout
        )
        return resp.json()["response"].strip(), (_time.perf_counter() - t0) * 1000.0
    llm_client.generate = _generate_with_system
    print(f'Legal-domain system prompt applied ({len(_LEGAL_SYSTEM_PROMPT)} chars).')

    t40_num10 = LLMJudgeReranker(
        first_stage_retriever=first_stage, article_texts=article_texts,
        llm_client=llm_client, top_n=TOP_N, max_article_tokens=1000,
        prompt_variant='0to10', cache_path=NUM10_CACHE,
        question_id_fn=lambda q: qid_map.get(q, hash(q) & 0x7FFFFFFF),
            fewshot_path=Path('evaluation/data/fewshot_examples.json'),
    )

    _orig_retrieve = t40_num10.retrieve
    _call_count    = [0]
    _t0_run        = time.perf_counter()

    def _retrieve_with_progress(query, top_k=10):
        ids, lat = _orig_retrieve(query, top_k=top_k)
        _call_count[0] += 1
        n = _call_count[0]
        if n % 20 == 0 or n == len(questions):
            elapsed = (time.perf_counter() - _t0_run) / 60
            eta     = elapsed / n * (len(questions) - n) if n < len(questions) else 0
            stats_  = t40_num10.get_stats()
            print(f'  [{n:3d}/{len(questions)}]  elapsed={elapsed:.1f}min  '
                  f'ETA={eta:.1f}min  parse_failures={stats_.get("parse_failure_rate", 0):.3f}')
        if n % CACHE_CHECKPOINT_EVERY == 0:
            t40_num10.save_cache()
            print(f'  [checkpoint] Cache saved ({n}/{len(questions)} questions done)')
        return ids, lat

    t40_num10.retrieve = _retrieve_with_progress

    print(f'\nRunning T4.0 0–10 numeric scoring (top_n={TOP_N}, ~3h on T4 GPU)...')
    t_exp  = time.perf_counter()
    result = run_experiment(
        retriever=t40_num10, questions=questions,
        experiment_id='llm_rerank_0to10_top50_test',
        hyperparameters={
            'first_stage': 'bm25_tuned_k11.5_b0.25', 'llm_backbone': 'llama3.1:8b',
            'llm_temperature': 0.0, 'top_n': TOP_N, 'max_article_tokens': 1000,
            'scoring_method': 'llm_judge_0to10', 'prompt_variant': '0to10',
            'fewshot_examples_file': 'evaluation/agentic/fewshot_examples.json',
            'bm25_k1': 1.5, 'bm25_b': 0.25,
            'bm25_normalization': 'lemmatize', 'bm25_field_weighting': 'text_only',
            'system_prompt_variant': 'legal_domain_context',
        },
        preprocessing={'normalization': 'lemmatize', 'field_weighting': 'text_only'},
        strata=strata, top_k=100,
    )
    exp_wall = time.perf_counter() - t_exp
    t40_num10.save_cache()

    stats      = t40_num10.get_stats()
    lbd        = t40_num10.get_latency_breakdown()
    score_diag = stats['score_diagnostics']

    result['index_build_time_s']        = round(index_build_s, 1)
    result['latency_breakdown_ms_mean'] = {
        'first_stage': lbd['first_stage_ms_mean'], 'llm_scoring': lbd['llm_scoring_ms_mean'],
    }
    result['llm_rerank_stats'] = {
        'mean_llm_calls_per_query': TOP_N, 'cache_size_after_run': stats['cache_size'],
        'parse_failure_rate': stats['parse_failure_rate'],
    }
    result['score_diagnostics'] = {
        'mean_score_per_query_mean':       score_diag['mean_score_per_query_mean'],
        'score_std_per_query_mean':        score_diag['score_std_per_query_mean'],
        'fraction_queries_all_same_score': score_diag['fraction_queries_all_same_score'],
        'llm_score_vs_ce_score_spearman_rho': None,
    }
    result['total_experiment_wall_clock_s'] = round(exp_wall, 1)
    result['significance_vs_anchor'] = {
        'anchor_experiment_id': 'bm25_tuned_k11.5_b0.25',
        'p_value_recall10': None, 'significant': None,
        'note': 'Appendix result — significance test not computed',
    }

    saved = save_result(result, results_dir=RESULTS_DIR)
    m     = result['metrics']
    print(f'\nSaved {saved}')
    print(f"  R@10={m.get('Recall@10',0):.4f}  MRR@10={m.get('MRR@10',0):.4f}  "
          f"parse_failures={stats['parse_failure_rate']:.4f}")
    print(f"  Total wall clock: {exp_wall/60:.1f} min")

In [ ]:
# ── Cell 12: Final results summary ───────────────────────────────────────────
import json
from pathlib import Path

# Scan both result subdirectories
result_dirs = {
    'binary/val+test': Path(f'{REPO_DIR}/output/results/agentic/llm_judge/llm_rerank'),
    '0to10/test':        Path(f'{REPO_DIR}/output/results/agentic/llm_judge/0to10'),
}

print(f'{"Experiment":<44}  R@10    R@100   MRR@10  sig')
print('-' * 78)
for label, results_dir in result_dirs.items():
    if not results_dir.exists():
        print(f'  [{label}] directory not found: {results_dir}')
        continue
    all_files = sorted(results_dir.glob('llm_rerank_*.json'))
    if not all_files:
        print(f'  [{label}] no result files found')
        continue
    for f in all_files:
        d   = json.loads(f.read_text())
        m   = d['metrics']
        sig = d.get('significance_vs_anchor', {}).get('significant', '-')
        p   = d.get('significance_vs_anchor', {}).get('p_value_recall10', None)
        p_str = f'p={p:.4f}' if p is not None else ''
        print(f"  {d['experiment_id']:<42}  {m['Recall@10']:.4f}  "
              f"{m['Recall@100']:.4f}  {m['MRR@10']:.4f}  {sig}  {p_str}")

In [ ]:
# ── Cell 13: Upload results to Azure Blob Storage ─────────────────────────────
# output/ is gitignored — results are NOT committed to git.
# On Azure, upload them to the blob container so they can be downloaded locally
# (via the local data root sync or manual download). The SAS URL must have Write permission.

import sys
from pathlib import Path
from azure.storage.blob import ContainerClient

RESULT_DIRS = {
    'results/agentic/llm_judge/llm_rerank': Path(f'{REPO_DIR}/output/results/agentic/llm_judge/llm_rerank'),
    'results/agentic/llm_judge/0to10':        Path(f'{REPO_DIR}/output/results/agentic/llm_judge/0to10'),
}

client   = ContainerClient.from_container_url(AZURE_CONTAINER_SAS_URL)
uploaded = []

for blob_prefix, results_dir in RESULT_DIRS.items():
    if not results_dir.exists():
        print(f'  [{blob_prefix}] directory not found — skipping.')
        continue
    for json_file in sorted(results_dir.glob('*.json')):
        blob_name = f'{blob_prefix}/{json_file.name}'
        print(f'  Uploading {json_file.name} → {blob_name} ...', end='', flush=True)
        with open(json_file, 'rb') as f:
            client.get_blob_client(blob_name).upload_blob(f, overwrite=True)
        print(' done')
        uploaded.append(blob_name)

print(f'\nUploaded {len(uploaded)} result file(s) to blob storage.')
print('Download locally:')
print('  python scripts/setup/download_results_from_blob.py  (if implemented)')
print('  — or — use Azure Storage Explorer to copy to output/results/agentic/llm_judge/')